# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata
print(f"Dataset Name: {metadata_obj.name}\nDescription: {metadata_obj.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s (unique identifiers). For every entity in Croissant (record sets, fields/columns), we always reference by `@id`.

In [ ]:
# List all available RecordSets by @id and name
recordsets = [rs for rs in dataset.record_sets]

print(f"Found {len(recordsets)} record set(s):\n")
for rs in recordsets:
    print(f"  RecordSet: @id={rs['@id']}  name={rs.get('name', '<no name>')}")

print("\n---\n")
# For each RecordSet, print the fields/columns and their `@id`s
for rs in recordsets:
    print(f"Fields/columns for RecordSet @id={rs['@id']}:")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"  - Field: @id={field['@id']}  name={field.get('name', '<no name>')}")
    print('')

## 3. Data Extraction
Load data from specific record set(s) into pandas DataFrame(s) for analysis using their `@id`.

Below, we extract data from all available RecordSets and preview the first few columns for the first RecordSet.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("RecordSet @ids:", record_set_ids)

dataframes = {}
for rs_id in record_set_ids:
    # Use mlcroissant's records() method with the record set @id
    print(f"Loading records for RecordSet @id={rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Loaded DataFrame for RecordSet {rs_id} with shape {df.shape}")
    else:
        print(f"  No records found for {rs_id}")

# For demonstration: preview columns for the first recordset with available data
for rs_id, df in dataframes.items():
    print(f"\nColumns in DataFrame for RecordSet {rs_id}:\n{df.columns.tolist()}")
    display(df.head())
    break  # Just preview the first loaded recordset

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on numeric fields, normalizing, and grouping. All columns and operations will reference by their `@id` and match those printed above.

In [ ]:
# Example EDA: choose a record set and a numeric field by their @id
# Update these IDs after inspecting the columns in section 2/3

# For demonstration, automatically select the first numeric column in first DataFrame
import numpy as np

# Select first non-empty record set
if len(dataframes) == 0:
    raise RuntimeError("No tabular record sets found in the dataset.")
selected_rs_id = next(iter(dataframes))
df = dataframes[selected_rs_id]

# Try to pick a likely numeric column
numeric_field = None
for col in df.columns:
    # Try to select columns with numeric dtype or convertible strings
    try:
        converted = pd.to_numeric(df[col], errors='coerce')
        if converted.notna().sum() > 0:
            numeric_field = col
            break
    except Exception:
        continue
if numeric_field is None:
    raise RuntimeError('No numeric-like column found for analysis.')

# Filter: keep records with numeric_field > threshold
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')  # ensure numeric
threshold = df[numeric_field].mean()  # for example, use mean as threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records in RecordSet {selected_rs_id} where {numeric_field} > mean (%.3f):" % threshold)
print(filtered_df[[numeric_field]].head())

# Normalize numeric_field within filtered records
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field}:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by another categorical column (if any string/object column exists)
group_field = None
for col in df.columns:
    if col == numeric_field:
        continue
    if df[col].dtype == object:
        nunique = df[col].nunique()
        if 1 < nunique < max(20, len(df)//10):  # avoid id-like columns
            group_field = col
            break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field} and averaged {numeric_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Histogram of selected numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field].dropna(), kde=True, color='royalblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# 2. Boxplot grouped by group_field (if found)
if group_field:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df, palette='Set2')
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load, explore, and analyze the FAIR^2 dataset on mass spectrometry analysis of extracellular vesicles in human mast cells using record set and field `@id`s. With these tools and this standard approach, you can efficiently load, filter, visualize, and downstream-process FAIR datasets described by a Croissant schema.